# Notebook 02b — Manual Validation of the Zero-Shot Classifier

**Purpose.** Validate the four classifier-generated scores (`benign_envy`, `malicious_envy`, `psi`, `purchase_intent`) against human judgement on a stratified sample of 200 comments. Without this validation, the downstream regression results from Notebook 03 cannot be defended against the fundamental measurement-validity question: how does the analyst know that the `malicious_envy` column is actually measuring malicious envy?

**The methodological key — blind coding.** The 200 comments must be hand-coded *without seeing the classifier's scores*. Anchoring bias otherwise drives judgements toward the classifier's output and the resulting F1 metrics are inflated. This notebook splits the work into two files:

| File | Contains | Purpose |
|---|---|---|
| `validation_coding_sheet.csv` | Comment text + empty judgment columns | Opened in a spreadsheet application for hand-coding |
| `validation_sample_full.csv` | Same comments + classifier scores | Remains closed until coding is finished |

**The four constructs to be coded.** For each comment a 0 or 1 is assigned to four columns:

- `human_benign_envy`: 1 if the comment expresses upward aspiration toward the influencer ("goals", "I want to be like her", "so inspiring"); 0 otherwise.
- `human_malicious_envy`: 1 if the comment expresses hostility, contempt, sarcasm, or resentment toward the influencer personally ("so fake", "must be nice", mocking remarks); 0 otherwise.
- `human_psi`: 1 if the comment treats the influencer as a personal friend, defends them, or expresses warm familiarity ("we love her", "leave my girl alone"); 0 otherwise.
- `human_purchase_intent`: 1 if the comment expresses an urgent desire to buy a specific product ("where's the link", "take my money", "need this immediately"); 0 otherwise.

A single comment can score 1 on multiple constructs (e.g., a malicious comment that also mentions wanting to buy a product can be 1 on both `human_malicious_envy` and `human_purchase_intent`).


## 1. Setup & load the scored corpus


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix,
    accuracy_score,
    cohen_kappa_score,
)

scored = pd.read_csv("comments_scored.csv")
print(f"Scored corpus: {len(scored):,} comments")
print(f"Tier balance: {scored['influencer_tier'].value_counts().to_dict()}")


## 2. Stratified random sample (n = 200)

The sample draws 100 comments from each tier (Mega, Micro) so the validation set has equal tier representation. Within each tier the draw is random — pure random sampling naturally provides spread across the score distribution, and stratifying further (e.g., by score quartile) adds complexity without materially improving the F1 estimate.

The seed is fixed (`random_state=42`) so the sample is reproducible. Re-generating the sample with the same seed returns the same 200 comments.


In [ ]:
SAMPLE_SIZE = 200

mega  = scored[scored["influencer_tier"] == "mega" ].sample(SAMPLE_SIZE//2, random_state=42)
micro = scored[scored["influencer_tier"] == "micro"].sample(SAMPLE_SIZE//2, random_state=42)

# Concatenate then shuffle so coding order isn't tier-blocked
sample = (pd.concat([mega, micro], ignore_index=True)
            .sample(frac=1, random_state=99)
            .reset_index(drop=True))

print(f"Sample size: {len(sample)}")
print(f"Tier balance: {sample['influencer_tier'].value_counts().to_dict()}")
print(f"\nClassifier-score quartile spread in the sample:")
for c in ["benign_envy", "malicious_envy", "psi", "purchase_intent"]:
    q = pd.qcut(sample[c], 4, labels=["Q1","Q2","Q3","Q4"]).value_counts().sort_index()
    print(f"  {c}: {q.to_dict()}")


## 3. Write the blind coding sheet and the full sample file

Two CSVs are written:

- `validation_coding_sheet.csv` — what you open in Excel. Contains comment text + 4 empty columns for your 0/1 judgements. **Does not include classifier scores.**
- `validation_sample_full.csv` — sealed reference file with the classifier scores. Do not open it until you've finished coding.


In [ ]:
# 3a. The BLIND coding sheet — no classifier scores visible
blind_cols = ["id", "matched_influencer", "influencer_tier", "subreddit", "body"]
blind = sample[blind_cols].copy()
blind["human_benign_envy"]     = ""
blind["human_malicious_envy"]  = ""
blind["human_psi"]             = ""
blind["human_purchase_intent"] = ""

blind.to_csv("validation_coding_sheet.csv", index=False)
print(f"Wrote validation_coding_sheet.csv  ({len(blind)} rows × {len(blind.columns)} cols)")

# 3b. The FULL sample file with classifier scores — for analysis after coding
sample.to_csv("validation_sample_full.csv", index=False)
print(f"Wrote validation_sample_full.csv  (do not open until coding is done)")


## 4. STOP HERE — go code the 200 comments

**Do not run any cells below this until you've finished coding.**

### Coding workflow

1. Open `validation_coding_sheet.csv` in Excel (or Numbers / Google Sheets).
2. For each row, read the `body` column carefully and assign 0 or 1 to each of the four `human_*` columns.
3. Save the file — keep it as a CSV (Excel may want to convert to .xlsx; resist that). When prompted, choose "Keep current format" or "CSV UTF-8".
4. Come back to this notebook and run the cells below.

### Coding rubric — apply these consistently

**`human_benign_envy = 1` if and only if** the commenter expresses *upward aspiration toward the influencer as a person*. They want to *be like* them, treat them as a role model, or are clearly *inspired* by them. Generic positive sentiment about a product is NOT benign envy. Vague compliments ("she's nice") without aspiration are NOT benign envy.
- **Examples → 1**: "absolute goals 😍", "she's everything I want to be", "her glow up is so inspiring", "I want her wardrobe so badly"
- **Examples → 0**: "this product is great", "interesting video", "I bought this last week", a hostile or sarcastic comment

**`human_malicious_envy = 1` if and only if** the commenter expresses *hostility, contempt, mockery, or bitter resentment toward the influencer as a person*. Criticism of a product is NOT malicious envy. Factual disagreement is NOT malicious envy. The hostility must be directed at the influencer personally.
- **Examples → 1**: "so fake", "must be nice 🙄", "she's such a fraud", "this is exhausting", "Jaclyn Hill 2.0 with all her bullshit"
- **Examples → 0**: "this lipstick has bad pigment", "I disagree with her review", a positive comment, a neutral observation

**`human_psi = 1` if and only if** the commenter writes about the influencer as if they personally know them — uses nicknames, defends them, expresses personal affection, refers to them as "my girl" or "we love her". Vague positive sentiment ("she's great") is NOT PSI; PSI requires expressed *closeness*, not just approval.
- **Examples → 1**: "we love her", "leave my girl alone", "Alix would never", "omg my queen", "I feel like I grew up with her"
- **Examples → 0**: "her makeup is good", "I watched her video", a hostile or critical comment

**`human_purchase_intent = 1` if and only if** the commenter expresses an *urgent desire to buy a specific product right now*. Discussing past purchases, browsing intentions, or general product conversation is NOT purchase intent — there must be expressed buying urgency in the present.
- **Examples → 1**: "where is the link?", "take my money", "adding to cart immediately", "I need this RIGHT NOW", "shut up and take my money"
- **Examples → 0**: "I bought this last month", "looks nice", "might try this someday", anti-consumption comments

### Tips for consistency

- If you're unsure on a comment, default to 0. Better to have a few false negatives in your hand-coding than to inflate every construct's recall by guessing.
- After coding the first 20, scroll back and check: are you applying the same standard? Calibrate yourself.
- Take a 10-minute break every 50 comments. Tired coders are inconsistent coders.
- Do not re-read a comment to "fix" it after seeing other rows — that's how rubric-drift sneaks in.

When the file is saved with all 200 rows coded, run the cells below.


## 5. Load the completed coding sheet

This cell expects `validation_coding_sheet.csv` to now contain your 0/1 judgements in the four `human_*` columns. We merge it back with the classifier scores from `validation_sample_full.csv`.


In [ ]:
coded = pd.read_csv("validation_coding_sheet.csv")
full  = pd.read_csv("validation_sample_full.csv")

# Sanity: make sure all 200 rows are coded with valid 0/1 values
constructs = ["benign_envy", "malicious_envy", "psi", "purchase_intent"]
human_cols = [f"human_{c}" for c in constructs]

# Coerce to numeric and check
for col in human_cols:
    coded[col] = pd.to_numeric(coded[col], errors="coerce")

n_missing = coded[human_cols].isna().any(axis=1).sum()
n_invalid = ((coded[human_cols] != 0) & (coded[human_cols] != 1)).any(axis=1).sum()
print(f"Rows with missing judgement: {n_missing}")
print(f"Rows with invalid (non-0/1) judgement: {n_invalid}")
if n_missing or n_invalid:
    print("\n⚠️  Fix the coding sheet and re-run this cell. All rows must have 0 or 1 in every human_ column.")

# Merge with classifier scores
merged = coded.merge(full[["id"] + constructs], on="id", suffixes=("", "_clf"))
# After merge: human_<construct> = your judgement; <construct> = classifier score
print(f"\nMerged dataset: {len(merged)} rows")
print(merged[["body"] + human_cols + constructs].head(3))


## 6. Compute precision, recall, F1, accuracy, Cohen's kappa per construct

For each construct the classifier score is thresholded at 0.5 (the conventional default for binary decisions on probability outputs) and compared to the human binary judgement. The four metrics conventionally expected in content-analysis validation are reported.

**Reading the metrics:**

- **Precision** — when the classifier said 1, how often was the human also 1? Low precision = many false positives.
- **Recall** — when the human said 1, how often did the classifier also say 1? Low recall = many false negatives.
- **F1** — harmonic mean of precision and recall. The primary summary metric.
- **Accuracy** — overall agreement rate (correct predictions / total). Misleading when classes are imbalanced.
- **Cohen's kappa** — agreement corrected for chance. > 0.6 is "substantial agreement", > 0.8 is "almost perfect" (Landis & Koch, 1977).

**Decision rules for the analysis** (field conventions, not arbitrary):

| F1 range | Interpretation | Treatment in the dissertation |
|---|---|---|
| F1 ≥ 0.75 | Strong validity | Construct used as a confirmatory variable in the regression. |
| 0.65 ≤ F1 < 0.75 | Acceptable validity | Construct used; F1 noted in Methods and findings treated as moderately strong. |
| 0.50 ≤ F1 < 0.65 | Marginal validity | Construct used; findings reported as exploratory. Flagged prominently in Limitations. |
| F1 < 0.50 | Poor validity | Construct not reliably measured. Dropped from the analysis or subjected to LLM-based re-classification. |


In [ ]:
THRESHOLD = 0.5

rows = []
for c in constructs:
    y_true = merged[f"human_{c}"].astype(int).values
    y_pred = (merged[c] >= THRESHOLD).astype(int).values

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    n_human_pos = int(y_true.sum())
    n_clf_pos = int(y_pred.sum())

    rows.append({
        "construct": c,
        "n_human_pos": n_human_pos,
        "n_clf_pos": n_clf_pos,
        "precision": round(p, 3),
        "recall": round(r, 3),
        "f1": round(f1, 3),
        "accuracy": round(acc, 3),
        "cohen_kappa": round(kappa, 3),
    })

metrics = pd.DataFrame(rows)
print("Per-construct validation metrics (threshold = 0.50):")
print(metrics.to_string(index=False))


## 7. Confusion matrices — see exactly where the disagreements are

Each confusion matrix shows, for one construct, the four possible outcomes:

|  | Classifier says 0 | Classifier says 1 |
|---|---|---|
| **Human says 0** | True Negative | False Positive |
| **Human says 1** | False Negative | True Positive |

A diagonal-heavy matrix is good (lots of TN and TP, few FP and FN). An off-diagonal-heavy matrix means the classifier is systematically disagreeing with you in one direction.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, c in zip(axes, constructs):
    y_true = merged[f"human_{c}"].astype(int).values
    y_pred = (merged[c] >= THRESHOLD).astype(int).values
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["clf=0","clf=1"], yticklabels=["human=0","human=1"], ax=ax)
    ax.set_title(c)
plt.suptitle(f"Confusion matrices at threshold {THRESHOLD}", y=1.05)
plt.tight_layout()
plt.show()


## 8. Threshold sensitivity — is 0.5 the right cutoff?

The default threshold of 0.5 may not be optimal for every construct. If a construct has a long-tailed score distribution (e.g. `purchase_intent`, where most comments score near 0 and only the strongest spikes pass 0.9), a lower threshold can recover more true positives. We sweep across thresholds and report the threshold that maximizes F1 per construct. Use the optimal threshold in your final analysis if it's substantially better than 0.5 — otherwise stick with 0.5 for simplicity and pre-registration honesty.


In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_results = []
for c in constructs:
    y_true = merged[f"human_{c}"].astype(int).values
    best_f1, best_t = 0, 0.5
    curve = []
    for t in thresholds:
        y_pred = (merged[c] >= t).astype(int).values
        _, _, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0)
        curve.append((t, f1))
        if f1 > best_f1:
            best_f1, best_t = f1, t
    threshold_results.append({"construct": c, "best_threshold": round(best_t, 2),
                              "best_f1": round(best_f1, 3),
                              "f1_at_0.5": round([f for t,f in curve if abs(t-0.5)<1e-6][0], 3)})
    plt.plot([t for t,f in curve], [f for t,f in curve], marker="o", label=c)

plt.axvline(0.5, color="grey", linestyle="--", alpha=0.5)
plt.xlabel("threshold")
plt.ylabel("F1")
plt.title("F1 vs. classifier threshold, per construct")
plt.legend()
plt.tight_layout()
plt.show()

print(pd.DataFrame(threshold_results).to_string(index=False))


## 9. Save the validation results and decide

The validation metrics are persisted so the exact F1 numbers can be cited in the Methods chapter without re-running the notebook. The decision then follows the F1 thresholds above: which constructs pass validity and proceed to Notebook 03, which need flagging in Limitations, and which (if any) need re-measurement.


In [ ]:
metrics.to_csv("validation_metrics.csv", index=False)
print("Saved validation_metrics.csv")
print()

# Auto-generate a decision summary
print("=" * 70)
print("VALIDATION DECISION SUMMARY")
print("=" * 70)
for _, row in metrics.iterrows():
    f1 = row["f1"]
    if f1 >= 0.75:
        verdict = "STRONG validity — confirmatory use OK."
    elif f1 >= 0.65:
        verdict = "ACCEPTABLE validity — use but report F1 in methods."
    elif f1 >= 0.50:
        verdict = "MARGINAL validity — exploratory only, flag in limitations."
    else:
        verdict = "POOR validity — drop or re-classify with stronger model."
    print(f"  {row['construct']:>18s}: F1={f1:.2f}, κ={row['cohen_kappa']:.2f}  →  {verdict}")
print()
